# St. Paul, MN parcels

Build a St. Paul-only export from Ramsey County OpenData parcels (ArcGIS FeatureServer layer 12).


In [1]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
from shapely.ops import unary_union

import sys
sys.path.append("..")
from cloud_utils import get_feature_data_with_geometry, ensure_geodataframe
from parcel_calculations import add_improvement_ratio_fields

# Config
SCRAPE_DATA = 0 # set to 1 to pull a fresh scrape from ArcGIS
DATA_DIR = "data/st_paul"
os.makedirs(DATA_DIR, exist_ok=True)

base_url = "https://maps.co.ramsey.mn.us/arcgis/rest/services/OpenData/OpenData/FeatureServer"
dataset_name = "12/query"
layer_id = 12

In [2]:
from cloud_utils import get_feature_data_with_geometry, ensure_geodataframe

if SCRAPE_DATA == 1:
    # Try robust chunked/paginated downloader using built-in cloud_utils
    print("🔄 Downloading full parcel set from ArcGIS paginated endpoint...")

    parcel_gdf = get_feature_data_with_geometry(dataset_name, base_url, layer_id, paginate=True)
    if parcel_gdf is None or len(parcel_gdf) == 0:
        raise RuntimeError("No parcels were downloaded from ArcGIS endpoint.")

    today_str = datetime.now().strftime("%Y_%m_%d")
    out_path = os.path.join(DATA_DIR, f"ramsey_parcels_{today_str}.parquet")
    parcel_gdf.to_parquet(out_path, index=False)
    print(f"✅ Saved new scrape to {out_path}")
else:
    files = glob.glob(os.path.join(DATA_DIR, "ramsey_parcels_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parcel files found in {DATA_DIR}. Set SCRAPE_DATA=1 to scrape.")

    files_sorted = sorted(
        files,
        key=lambda x: datetime.strptime(
            os.path.basename(x).replace("ramsey_parcels_", "").replace(".parquet", ""),
            "%Y_%m_%d"
        ),
        reverse=True,
    )
    latest_file = files_sorted[0]
    print(f"✅ Loading most recent scrape: {latest_file}")
    parcel_gdf = pd.read_parquet(latest_file)

parcel_gdf = ensure_geodataframe(parcel_gdf)
print(f"✅ Loaded as {type(parcel_gdf).__name__} | CRS={parcel_gdf.crs} | rows={len(parcel_gdf):,}")


✅ Loading most recent scrape: data/st_paul/ramsey_parcels_2026_02_13.parquet


✅ Loaded as GeoDataFrame | CRS=EPSG:4326 | rows=167,677


In [3]:
# St. Paul filter
city_series = (
    parcel_gdf.get("SiteCityName")
    .fillna(parcel_gdf.get("SiteCityNameUSPS"))
    .astype(str)
    .str.upper()
    .str.strip()
)
parcel_gdf = parcel_gdf[city_series == "SAINT PAUL"].copy()

print(f"✅ Rows after SiteCityName == 'SAINT PAUL': {len(parcel_gdf):,}")


✅ Rows after SiteCityName == 'SAINT PAUL': 83,399


In [4]:
# Filter out fully tax-exempt parcels
st_paul_gdf = parcel_gdf.copy()
st_paul_gdf['fully_exempt'] = (st_paul_gdf['TaxExemptYN'] == 'Y')

before_count = len(st_paul_gdf)
parcel_gdf = st_paul_gdf[~st_paul_gdf['fully_exempt']].copy()
after_count = len(parcel_gdf)

print(f"✅ Removed {before_count - after_count:,} fully exempt parcels (TaxExemptYN == 'Y')")
print(f"✅ Rows remaining after exemption filter: {after_count:,}")

✅ Removed 3,408 fully exempt parcels (TaxExemptYN == 'Y')
✅ Rows remaining after exemption filter: 79,991


In [5]:
# Condo cure: collapse duplicate parcel groups similar to other jurisdiction examples.
# Prefer FIPsCodeParcelID, then ParcelID.
parcel_gdf["parcel_group_id"] = (
    parcel_gdf.get("FIPsCodeParcelID").fillna(parcel_gdf.get("ParcelID")).astype(str).str.strip()
)

dup_count = parcel_gdf.duplicated(subset=["parcel_group_id"], keep=False).sum()
print(f"Duplicate rows by parcel_group_id before collapse: {dup_count}")

numeric_sum_candidates = [
    "EMVLand", "EMVBuilding", "EMVTotal", "TotalTax", "SpecialAssessmentDue", "TaxCapacity",
    "ParcelSquareFeet", "ParcelAcresDeed", "ParcelAcresPolygon",
    "EMVLand1", "EMVBuilding1", "EMVTotal1", "TotalTax1", "SpecialAssessmentDue1",
    "EMVLand2", "EMVBuilding2", "EMVTotal2", "TotalTax2", "SpecialAssessmentDue2",
]
numeric_sum_cols = [c for c in numeric_sum_candidates if c in parcel_gdf.columns]

categorical_cols = [
    c for c in parcel_gdf.columns
    if c not in set(numeric_sum_cols + ["geometry"])
]

agg_dict = {c: "sum" for c in numeric_sum_cols}
agg_dict.update({c: "first" for c in categorical_cols})
collapsed = parcel_gdf.groupby("parcel_group_id", dropna=False).agg(agg_dict).reset_index(drop=True)

# geometry union per group
geom_union = parcel_gdf.groupby("parcel_group_id", dropna=False)["geometry"].apply(
    lambda geoms: unary_union([g for g in geoms if g is not None]) if any(g is not None for g in geoms) else None
)
collapsed["geometry"] = geom_union.values

parcel_gdf = gpd.GeoDataFrame(collapsed, geometry="geometry", crs=parcel_gdf.crs)
print(f"✅ Rows after condo cure collapse: {len(parcel_gdf):,}")


Duplicate rows by parcel_group_id before collapse: 0


✅ Rows after condo cure collapse: 79,991


In [6]:
for idx, (desc, count) in enumerate(parcel_gdf["LandUseCodeDescription"].value_counts().items(), 1):
    print(f"{desc:40} {count}")

SINGLE FAMILY DWELLING, PLATTED LOT      57478
CONDO                                    5657
TWO FAMILY DWELLING - UP/DWN             4494
CONDO GARAGE                             1164
RESIDENTIAL, VACANT LAND, LOT            1103
TWO FAMILY DWELLING - SIDE/SIDE          890
COMMERCIAL VACANT LAND                   684
APARTMENTS 4-6 RENTAL UNITS              677
APARTMENTS 7-19 RENTAL UNITS             667
TOWNHOME-INNER UNIT                      624
TOWNHOME - OUTER UNIT                    557
MIXED RESID/COMMERCIAL                   508
TOWNHOME - TICO                          399
THREE FAMILY DWELLING, PLATTED LOT       394
IND WAREHOUSE                            363
SMALL (UNDER 10,000 SF) DET. RETAIL      329
RESIDENTIAL CO-OP                        328
INDUSTRIAL, VACANT LAND                  297
OFFICE BUILDING 1-2 STORIES              268
APARTMENTS 20-49 RENTAL UNITS            267
TWIN HOME                                250
APARTMENT VACANT LAND                    196
AUTO

In [7]:
def categorize_property_type(land_use_desc):
    s = str(land_use_desc).upper()
    if any(k in s for k in ["VACANT", "UNPLATTED"]):
        return "Vacant Land"
    if any(k in s for k in ["SINGLE FAMILY", "HOMESTEAD"]):
        return "Single Family"
    if any(k in s for k in ["CONDO", "TOWNHOUSE", "APARTMENT", "DUPLEX", "TRIPLEX", "MULTI"]):
        return "Multi-Family / Condo"
    if any(k in s for k in ["INDUSTR", "WAREHOUSE", "MANUFACTUR"]):
        return "Manufacturing/Industrial"
    if any(k in s for k in ["COMMERCIAL", "RETAIL", "OFFICE", "HOTEL", "MOTEL"]):
        return "Retail/Service/Commercial"
    if any(k in s for k in ["EXEMPT", "CHURCH", "SCHOOL", "GOV"]):
        return "Institutional / Exempt"
    return "Other"

parcel_gdf["PROPERTY_CATEGORY"] = parcel_gdf["LandUseCodeDescription"].apply(categorize_property_type)


In [8]:
export_gdf = parcel_gdf.copy()

# Exemption flag
export_gdf["exemption_flag"] = export_gdf.get("TaxExemptYN", "N").astype(str).str.upper().eq("Y").astype(int)

# Core values
export_gdf["land_value"] = pd.to_numeric(export_gdf.get("EMVLand"), errors="coerce")
export_gdf["improvement_value"] = pd.to_numeric(export_gdf.get("EMVBuilding"), errors="coerce")
export_gdf["full_market_value"] = pd.to_numeric(export_gdf.get("EMVTotal"), errors="coerce")

# Area
export_gdf["area_sqft"] = pd.to_numeric(export_gdf.get("ParcelSquareFeet"), errors="coerce")
if export_gdf["area_sqft"].isna().all() or (export_gdf["area_sqft"] <= 0).all():
    area_proj = export_gdf.to_crs(26915).geometry.area * 10.7639104167
    export_gdf["area_sqft"] = area_proj
export_gdf["area_sqft"] = export_gdf["area_sqft"].replace(0, np.nan)

export_gdf["full_market_value_per_sqft"] = export_gdf["full_market_value"] / export_gdf["area_sqft"]
export_gdf["land_value_per_sqft"] = export_gdf["land_value"] / export_gdf["area_sqft"]
export_gdf["improvement_value_per_sqft"] = export_gdf["improvement_value"] / export_gdf["area_sqft"]

export_gdf["property_land_use_category"] = export_gdf["PROPERTY_CATEGORY"]

def categorize_property_refined(row):
    cat = str(row["PROPERTY_CATEGORY"])
    if "Vacant" in cat:
        return "Vacant"
    if "Condo" in cat:
        return "Condo"
    if row["improvement_value"] < 0.5 * (row["land_value"] + row["improvement_value"]):
        return "Underdeveloped"
    return None

export_gdf["property_land_use_refined"] = export_gdf.apply(categorize_property_refined, axis=1)

export_gdf = add_improvement_ratio_fields(
    export_gdf,
    land_col="land_value",
    improvement_col="improvement_value",
)

if "link" not in export_gdf.columns:
    export_gdf["link"] = (
        "https://www.ramseycounty.us/residents/property-home/property-tax-and-value-lookup?pid="
        + export_gdf.get("ParcelID", "").astype(str)
    )


In [9]:
columns_to_export = [
    "geometry",
    "exemption_flag",
    "property_land_use_category",
    "property_land_use_refined",
    "full_market_value",
    "full_market_value_per_sqft",
    "land_value",
    "land_value_per_sqft",
    "improvement_value",
    "improvement_value_per_sqft",
    "TLLDIMPROV",
    "IMPR_LAND_RATIO",
    "IMPR_LAND_PCT",
    "IMPR_PCT_TOTAL",
    "link",
]

for col in columns_to_export:
    if col not in export_gdf.columns:
        export_gdf[col] = np.nan

export_final = export_gdf[columns_to_export].rename(columns={
    "land_value": "current_full_land_value"
})

export_final["geometry"] = export_final["geometry"].apply(
    lambda geom: geom if geom is None or geom.is_valid else geom.buffer(0)
)

export_final = gpd.GeoDataFrame(export_final, geometry="geometry", crs=export_gdf.crs)
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.to_crs("EPSG:4326")

canonical_path = os.path.join(DATA_DIR, "st-paul-mn-parcels.parquet")
today_str = datetime.now().strftime("%Y_%m_%d")
dated_path = os.path.join(DATA_DIR, f"st-paul-mn-parcels_{today_str}.parquet")

export_final.to_parquet(canonical_path, index=False)
export_final.to_parquet(dated_path, index=False)

print(f"✅ Saved export parquet: {canonical_path}")
print(f"✅ Also saved dated version: {dated_path}")
print(export_final.head())


✅ Saved export parquet: data/st_paul/st-paul-mn-parcels.parquet
✅ Also saved dated version: data/st_paul/st-paul-mn-parcels_2026_02_18.parquet
                                            geometry  exemption_flag  \
0  POLYGON ((-93.10651 44.94814, -93.10649 44.948...               0   
1  POLYGON ((-93.10655 44.94763, -93.10657 44.947...               0   
2  POLYGON ((-93.10671 44.94753, -93.10656 44.947...               0   
3  POLYGON ((-93.10668 44.94732, -93.10685 44.947...               0   
4  POLYGON ((-93.10909 44.94809, -93.10886 44.948...               0   

  property_land_use_category property_land_use_refined  full_market_value  \
0       Multi-Family / Condo                     Condo          2038800.0   
1       Multi-Family / Condo                     Condo           519400.0   
2       Multi-Family / Condo                     Condo           582600.0   
3       Multi-Family / Condo                     Condo          3090300.0   
4       Multi-Family / Condo           

In [10]:
# Optional: upload export_final to dev Azure blob
upload_dev = True

if upload_dev:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "Set AZURE_STORAGE_CONNECTION_STRING or update connection_string before upload."
        )

    container = os.getenv("AZURE_DEV_CONTAINER", "parquets-dev")
    blob_name = "st-paul-mn-parcels.parquet"
    local_path = os.path.join(DATA_DIR, blob_name)

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Local parquet not found: {local_path}")

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    container_client = blob_service.get_container_client(container)

    with open(local_path, "rb") as handle:
        container_client.upload_blob(name=blob_name, data=handle, overwrite=True)

    print(f"✅ Uploaded {local_path} -> {container}/{blob_name}")
else:
    print("upload_dev is False; skipping upload.")


✅ Uploaded data/st_paul/st-paul-mn-parcels.parquet -> parquets-dev/st-paul-mn-parcels.parquet


In [11]:
# Optional: promote dev blob to prod
promote_to_prod = True
promote_overwrite = True

if promote_to_prod:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "Set AZURE_STORAGE_CONNECTION_STRING or update connection_string before promotion."
        )

    dev_container = os.getenv("AZURE_DEV_CONTAINER", "parquets-dev")
    prod_container = os.getenv("AZURE_PROD_CONTAINER", "parquets-prod")
    blob_name = "st-paul-mn-parcels.parquet"

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    dev_blob = blob_service.get_blob_client(dev_container, blob_name)
    prod_blob = blob_service.get_blob_client(prod_container, blob_name)

    if not dev_blob.exists():
        raise FileNotFoundError(f"Dev blob not found: {dev_container}/{blob_name}")

    if prod_blob.exists():
        if not promote_overwrite:
            raise FileExistsError(
                "Prod blob already exists. Set promote_overwrite=True to replace it."
            )
        prod_blob.delete_blob()

    prod_blob.start_copy_from_url(dev_blob.url)
    print(f"✅ Promoted {dev_container}/{blob_name} -> {prod_container}/{blob_name}")
else:
    print("promote_to_prod is False; skipping promotion.")

✅ Promoted parquets-dev/st-paul-mn-parcels.parquet -> parquets-prod/st-paul-mn-parcels.parquet
